<a href="https://colab.research.google.com/github/blancavazquez/PLN/blob/main/notebooks/2_2_Embeddings_Word2vec_Familia_Simpsons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim

In [ ]:
import re  # For preprocessing
import pandas as pd  # For data handling
from time import time  # To time our operations
from collections import defaultdict  # For word frequency
import spacy  # For preprocessing
import numpy
import logging  # Setting up the loggings to monitor gensim
logging.basicConfig(format="%(levelname)s - %(asctime)s: %(message)s", datefmt= '%H:%M:%S', level=logging.INFO)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/1 IIMAS Mérida/Docs/2025_Clase_PLN/Slides/simpsons_dataset.csv')
print("Tamaño de la base de datos", df.shape)

In [ ]:
df.head()

In [ ]:
#Eliminando datos ausentes
df = df.dropna().reset_index(drop=True)
df.isnull().sum()

## Preprocesamiento

In [ ]:
nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser']) # disabling Named Entity Recognition for speed
def cleaning(doc):
    txt = [token.lemma_ for token in doc if not token.is_stop]
    if len(txt) > 2:
        return ' '.join(txt)

In [ ]:
#Removiendo caracteres especiales
brief_cleaning = (re.sub("[^A-Za-z']+", ' ', str(row)).lower() for row in df['spoken_words'])

In [ ]:
# Pipeline de spacy
t = time()
txt = [cleaning(doc) for doc in nlp.pipe(brief_cleaning, batch_size=5000)]
print('Time to clean up everything: {} mins'.format(round((time() - t) / 60, 2)))

In [ ]:
# Se almacenan los resultados en un dataframe
df_clean = pd.DataFrame({'clean': txt})
df_clean = df_clean.dropna().drop_duplicates()
df_clean.shape

In [ ]:
df_clean

In [ ]:
## Trabajando n-gramas
from gensim.models.phrases import Phrases, Phraser
sent = [row.split() for row in df_clean['clean']]
print("Tipo:", type(sent), "\n", sent)

In [ ]:
#Crea frases relevantes a partir de la lista de oraciones:
phrases = Phrases(sent, min_count=30, progress_per=10000)
bigram = Phraser(phrases)
sentences = bigram[sent]

In [ ]:
sentences

In [ ]:
#Gensim Word2Vec Implementation
import multiprocessing
from gensim.models import Word2Vec

In [ ]:
w2v_model = Word2Vec(min_count=20,
                     window=2,
                     sg= 0,   #1 skip-gram, 0 CBOW
                     vector_size = 300,
                     sample=6e-5,
                     alpha=0.03,
                     min_alpha=0.0007,
                     negative=20,)

In [ ]:
#Connstruyendo la tabla de vocabulario
t = time()
w2v_model.build_vocab(sentences, progress_per=10000)
print('Time to build vocab: {} mins'.format(round((time() - t) / 60, 2)))

In [ ]:
#Training
t = time()
w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)
print('Time to train the model: {} mins'.format(round((time() - t) / 60, 2)))

In [ ]:
w2v_model.init_sims(replace=True)

#Buscando elementos similares

In [ ]:
w2v_model.wv.most_similar(positive=["homer"])

In [ ]:
w2v_model.wv.most_similar(positive=["homer_simpson"])

In [ ]:
w2v_model.wv.similarity('maggie', 'baby')

In [ ]:
w2v_model.wv.doesnt_match(["nelson", "bart", "milhouse"])

In [ ]:
w2v_model.wv.doesnt_match(['homer', 'patty', 'selma'])

#Visualización

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
sns.set_style("darkgrid")

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
def tsnescatterplot(model, word, list_names):
    """ Plot in seaborn the results from the t-SNE dimensionality reduction algorithm of the vectors of a query word,
    its list of most similar words, and a list of words.
    """
    arrays = np.empty((0, 300), dtype='f')
    word_labels = [word]
    color_list  = ['red']

    # adds the vector of the query word
    arrays = np.append(arrays, model.wv.__getitem__([word]), axis=0)

    # gets list of most similar words
    close_words = model.wv.most_similar([word])

    # adds the vector for each of the closest words to the array
    for wrd_score in close_words:
        wrd_vector = model.wv.__getitem__([wrd_score[0]])
        word_labels.append(wrd_score[0])
        color_list.append('blue')
        arrays = np.append(arrays, wrd_vector, axis=0)

    # adds the vector for each of the words from list_names to the array
    for wrd in list_names:
        wrd_vector = model.wv.__getitem__([wrd])
        word_labels.append(wrd)
        color_list.append('green')
        arrays = np.append(arrays, wrd_vector, axis=0)

    # Reduces the dimensionality from 300 to 50 dimensions with PCA
    reduc = PCA(n_components=19).fit_transform(arrays)

    # Finds t-SNE coordinates for 2 dimensions
    np.set_printoptions(suppress=True)

    Y = TSNE(n_components=2, random_state=0, perplexity=15).fit_transform(reduc)

    # Sets everything up to plot
    df = pd.DataFrame({'x': [x for x in Y[:, 0]],
                       'y': [y for y in Y[:, 1]],
                       'words': word_labels,
                       'color': color_list})

    fig, _ = plt.subplots()
    fig.set_size_inches(9, 9)

    # Basic plot
    p1 = sns.regplot(data=df,
                     x="x",
                     y="y",
                     fit_reg=False,
                     marker="o",
                     scatter_kws={'s': 40,
                                  'facecolors': df['color']
                                 }
                    )

    # Adds annotations one by one with a loop
    for line in range(0, df.shape[0]):
         p1.text(df["x"][line],
                 df['y'][line],
                 '  ' + df["words"][line].title(),
                 horizontalalignment='left',
                 verticalalignment='bottom', size='medium',
                 color=df['color'][line],
                 weight='normal'
                ).set_size(15)


    plt.xlim(Y[:, 0].min()-50, Y[:, 0].max()+50)
    plt.ylim(Y[:, 1].min()-50, Y[:, 1].max()+50)

    plt.title('t-SNE visualization for {}'.format(word.title()))

In [ ]:
print("Visualizando las palabras más cercanas a Homero y ¿qué pasa si comparamos con palabras random?")
tsnescatterplot(w2v_model, 'homer', ['lisa', 'maggie', 'ah', 'maude', 'bob', 'mel', 'apu', 'duff'])

In [ ]:
print("10 palabras más similares vs. 10 más diferentes")
tsnescatterplot(w2v_model, 'homer', [i[0] for i in w2v_model.wv.most_similar(negative=["maggie"])])

In [ ]:
tsnescatterplot(w2v_model, "maggie", [t[0] for t in w2v_model.wv.most_similar(positive=["mr_burn"], topn=20)][10:])